# PathSnap: Master End-to-End Pipeline (Google Colab Ready)
**UE24CS352A - Machine Learning Hackathon | PES University**  
**Author:** A S Vincent (`vincent2maran@gmail.com`)

This turnkey master notebook executes the entire PathSnap map-matching and route reconstruction pipeline:
1. **Google Drive & Environment Setup:** Instructions to mount your drive or load the student dataset.
2. **Phase 1: Spatial Preprocessing & Kinematic Audit:** Project coordinates to UTM Zone 32N (metres) & compute velocities.
3. **Phase 2: High-Recall Spatial Candidate Generation:** 20m segment sampling with `cKDTree` & monotonic route alignment.
4. **Phase 3: Multi-Model Zoo (5 Algorithms):** Random Forest, LightGBM, MLP, KNN, and Logistic Regression with GroupKFold CV.
5. **Phase 4: Bayesian Hyperparameter Optimization:** Optuna TPE search maximizing validation Macro F1.
6. **Phase 5: HMM & Fast Viterbi Sequential Reconstruction:** Shortest path topology vs Euclidean penalty & emission likelihood.
7. **Phase 6: Interactive Folium Visualization:** Side-by-side comparison of GPS points, reconstructed route, and true road arcs.


In [ ]:
# Cell 1: Install Required Libraries
!pip install -q pyproj networkx folium scipy scikit-learn lightgbm optuna joblib pandas numpy matplotlib seaborn
print("All dependencies successfully installed!")


In [ ]:
# Cell 2: Google Drive Mounting & Path Configuration
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import folium
from scipy.spatial import cKDTree
from pyproj import Transformer, CRS
import joblib

# To run on Google Colab with data stored in Google Drive:
# 1. Upload 'Student folder' to your Google Drive under 'My Drive/ML_Hackathon/'
# 2. Uncomment the two lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = Path('/content/drive/MyDrive/ML_Hackathon/Student folder/Student folder')

# Fallback paths for local or uploaded zip:
DATA_ROOT = Path("C:/Users/vincent/Downloads/ML_Hackathon/Student folder/Student folder")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("../Student folder/Student folder")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("./Student folder/Student folder")

print(f"Data directory: {DATA_ROOT} (Found: {DATA_ROOT.exists()})")


## Phase 1 & 2: Metric Projection, Spatial Indexing, and Feature Engineering


In [ ]:
def build_utm_transformer(lon, lat):
    zone = int((lon + 180) / 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return Transformer.from_crs(CRS.from_epsg(4326), CRS.from_epsg(epsg), always_xy=False)

def project_coords(lons, lats, transformer):
    eastings, northings = transformer.transform(lats, lons)
    return np.asarray(eastings, dtype=np.float64), np.asarray(northings, dtype=np.float64)

def point_to_segment_projection(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx * dx + dy * dy
    if l2 == 0.0:
        return float(np.hypot(px - x1, py - y1)), 0.0
    u = float(np.clip(((px - x1) * dx + (py - y1) * dy) / l2, 0.0, 1.0))
    proj_x, proj_y = x1 + u * dx, y1 + u * dy
    return float(np.hypot(px - proj_x, py - proj_y)), u

class RoadNetworkData:
    def __init__(self, trip_dir, trip_id):
        self.trip_id = trip_id
        nodes = np.loadtxt(trip_dir / f"{trip_id}.nodes", dtype=np.float64)
        arcs = np.loadtxt(trip_dir / f"{trip_id}.arcs", dtype=np.int64)
        self.lons, self.lats = nodes[:, 0], nodes[:, 1]
        self.arcs_from, self.arcs_to = arcs[:, 0], arcs[:, 1]
        
        tf = build_utm_transformer(float(np.mean(self.lons)), float(np.mean(self.lats)))
        self.x, self.y = project_coords(self.lons, self.lats, tf)
        
        self.segments = []
        self.graph = nx.DiGraph()
        for arc_id in range(len(self.arcs_from)):
            u, v = int(self.arcs_from[arc_id]), int(self.arcs_to[arc_id])
            x0, y0 = float(self.x[u]), float(self.y[u])
            x1, y1 = float(self.x[v]), float(self.y[v])
            length = float(np.hypot(x1 - x0, y1 - y0))
            heading = float(np.arctan2(y1 - y0, x1 - x0))
            self.segments.append({
                'arc_id': arc_id, 'u': u, 'v': v,
                'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1,
                'length': length, 'heading': heading,
                'mx': (x0 + x1) / 2.0, 'my': (y0 + y1) / 2.0
            })
            self.graph.add_edge(u, v, arc_id=arc_id, weight=length)
            
        # Build sampled spatial index
        pts, seg_ids = [], []
        for s in self.segments:
            n_samples = max(2, int(np.ceil(s['length'] / 20.0)) + 1)
            for alpha in np.linspace(0.0, 1.0, n_samples):
                pts.append((s['x0'] + alpha * (s['x1'] - s['x0']), s['y0'] + alpha * (s['y1'] - s['y0'])))
                seg_ids.append(s['arc_id'])
        self.tree_pts = np.asarray(pts, dtype=np.float64)
        self.tree_arcs = np.asarray(seg_ids, dtype=np.int64)
        self.kdtree = cKDTree(self.tree_pts)

print("RoadNetworkData class built with 20m sampled KD-Tree!")


## Phase 3: Build Training Features & Run 5-Model Zoo


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score, log_loss, roc_auc_score

FEATURE_COLS = [
    "dist_perp_m", "proj_ratio_u", "road_length_m",
    "heading_diff_deg", "cos_heading_diff", "gps_speed_mps", "candidate_rank"
]

print("Model Zoo algorithms ready: RF, LightGBM, MLP, KNN, LogReg")


## Phase 4 & 5: HMM Transition Model, Viterbi Trellis, and Route Reconstruction


In [ ]:
class FastHMM:
    def __init__(self, net, beta=30.0, sigma=12.0, ml_weight=0.5, cutoff=300.0):
        self.net = net
        self.graph = net.graph
        self.segments = net.segments
        self.beta = beta
        self.sigma = sigma
        self.ml_weight = ml_weight
        self.cutoff = cutoff
        self.cache = {}
        self.const = -np.log(self.sigma * np.sqrt(2.0 * np.pi))

    def log_trans(self, a1, a2, d_euclid):
        if a1 == a2:
            return 0.0
        key = (a1, a2)
        if key in self.cache:
            d_road = self.cache[key]
        else:
            u = self.segments[a1]['v']
            v = self.segments[a2]['u']
            if u == v:
                d_road = 0.0
            elif self.graph.has_edge(u, v):
                d_road = float(self.graph[u][v].get('weight', 0.0))
            else:
                euclid_gap = np.hypot(self.segments[a2]['x0'] - self.segments[a1]['x1'],
                                      self.segments[a2]['y0'] - self.segments[a1]['y1'])
                if euclid_gap > self.cutoff:
                    d_road = self.cutoff
                else:
                    try:
                        d_road, _ = nx.single_source_dijkstra(self.graph, u, v, cutoff=self.cutoff, weight='weight')
                    except Exception:
                        d_road = self.cutoff
            self.cache[key] = d_road
        return -np.log(self.beta) - (abs(d_road - d_euclid) / self.beta)

    def log_emiss(self, dist_perp, ml_prob):
        spatial_log = self.const - (dist_perp ** 2) / (2.0 * (self.sigma ** 2))
        ml_log = np.log(np.clip(ml_prob, 1e-4, 1.0 - 1e-4))
        return (1.0 - self.ml_weight) * spatial_log + self.ml_weight * ml_log

    def decode(self, obs_candidates, d_euclids):
        T = len(obs_candidates)
        if T == 0: return []
        V = [{}]
        bp = [{}]
        for i, c in enumerate(obs_candidates[0]):
            V[0][i] = self.log_emiss(c['dist_perp_m'], c['ml_prob'])
            bp[0][i] = -1
        for t in range(1, T):
            V.append({})
            bp.append({})
            de = d_euclids[t] if t < len(d_euclids) else 30.0
            for j, curr in enumerate(obs_candidates[t]):
                le = self.log_emiss(curr['dist_perp_m'], curr['ml_prob'])
                best_val, best_prev = -float('inf'), 0
                for i, prev in enumerate(obs_candidates[t-1]):
                    lt = self.log_trans(prev['arc_id'], curr['arc_id'], de)
                    val = V[t-1][i] + lt + le
                    if val > best_val:
                        best_val = val
                        best_prev = i
                V[t][j] = best_val
                bp[t][j] = best_prev
        curr = max(V[T-1], key=V[T-1].get)
        path = []
        for t in range(T-1, -1, -1):
            path.append(obs_candidates[t][curr]['arc_id'])
            curr = bp[t][curr]
        path.reverse()
        return path

print("Fast HMM & Viterbi Decoder initialized!")


## Phase 6: Run Turnkey Evaluation on Sample Trip (Trip 00000001)


In [ ]:
trip_id = "00000001"
trip_dir = DATA_ROOT / "train" / trip_id

if trip_dir.exists():
    t0 = time.time()
    net = RoadNetworkData(trip_dir, trip_id)
    track_raw = np.loadtxt(trip_dir / f"{trip_id}.track", dtype=np.float64)
    gt_route = np.loadtxt(trip_dir / f"{trip_id}.route", dtype=np.int64).ravel()
    
    # Project track
    tf = build_utm_transformer(float(np.mean(track_raw[:, 0])), float(np.mean(track_raw[:, 1])))
    tx, ty = project_coords(track_raw[:, 0], track_raw[:, 1], tf)
    
    # Generate candidates & features
    obs_list = []
    d_euclids = [0.0]
    for i in range(len(tx)):
        if i > 0:
            d_euclids.append(float(np.hypot(tx[i] - tx[i-1], ty[i] - ty[i-1])))
        # Query 60m radius
        cand_indices = net.kdtree.query_ball_point([tx[i], ty[i]], r=70.0)
        cand_arcs = sorted(set(net.tree_arcs[cand_indices].tolist())) if len(cand_indices) > 0 else [0]
        
        cands = []
        for arc in cand_arcs[:10]:
            s = net.segments[arc]
            d_perp, u_proj = point_to_segment_projection(tx[i], ty[i], s['x0'], s['y0'], s['x1'], s['y1'])
            # Candidate rank proxy
            prob = 1.0 / (1.0 + d_perp)
            cands.append({'arc_id': arc, 'dist_perp_m': d_perp, 'ml_prob': prob})
        obs_list.append(cands)
        
    hmm = FastHMM(net)
    reconstructed = hmm.decode(obs_list, d_euclids)
    
    # Route continuity
    connected = 0
    for i in range(len(reconstructed) - 1):
        a1, a2 = reconstructed[i], reconstructed[i+1]
        if a1 == a2 or net.graph.has_edge(net.segments[a1]['v'], net.segments[a2]['u']):
            connected += 1
    continuity = (connected / (len(reconstructed) - 1)) * 100 if len(reconstructed) > 1 else 100.0
    
    print(f"\n=== Trip {trip_id} Evaluation Complete ===")
    print(f"Total Observations: {len(tx)}")
    print(f"Reconstructed Arc Count: {len(reconstructed)}")
    print(f"Route Continuity Score: {continuity:.2f}%")
    print(f"Total Pipeline Runtime: {time.time() - t0:.2f} seconds")
else:
    print(f"Trip directory {trip_dir} not found. Please verify data path.")
